# Laboratório 03 Preparação para sua implementação

Tarefa do professor: uma chain que indique positivo ou negativo para as avaliações da API. O trabalho é seu: este notebook organiza decisões e testes, sem preencher a implementação avaliada.

Defina antes de programar: política para comentários mistos, formato de saída, tratamento de erros e amostra com rótulos humanos.

O objetivo da classificaçao é a experiência geral do client.
A avaliação deve seguir a seguinte regra:

Se tiver apenas um aspecto segue a avaliação do aspecto

se tiver varios aspectos segue a maioria

em empate segue essa ordem: produto e atendimento

outro caso de empate negativo

agente seja exatamente positivo ou negativo.


Exemplos de avaliaçao
- "O produto é excelente" avaliação positiva
- "O produto é ótimo, o atendimento foi péssimo e a entrega atrasou." avaliaçao negativa
- "O produto veio quebrado, mas o atendimento foi excelente." avaliação negativa
- "A entrega foi rápida, mas a embalagem veio danificada." avaliação negativa

In [18]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

def construir_chain(modelo):
    system_prompt = """Classifique o sentimento da experiência geral do cliente.
Aplique as regras nesta ordem:
1. Considere apenas os aspectos mencionados; cada aspecto distinto conta uma vez.
2. Se houver um único aspecto, siga o sentimento dele.
3. Se houver vários aspectos, siga a maioria dos aspectos positivos ou negativos.
4. Apenas em empate, siga o sentimento do produto; se produto não for mencionado, siga o atendimento.
5. Se o empate continuar sem resolução, responda negativo.

Responda exatamente positivo ou negativo: uma palavra no masculino e em minúsculas.
Não inclua prefixos, explicações, aspas ou pontuação na resposta.

Exemplo 1:
Comentário: O suporte foi atencioso e resolveu minha dúvida.
Resposta: positivo

Exemplo 2:
Comentário: O aparelho não funciona, embora a equipe tenha sido gentil.
Resposta: negativo

Classifique o próximo comentário seguindo as mesmas regras.
Sua resposta deve conter somente positivo ou negativo."""

    human_prompt = "Comentario: {comentario}"
    prompt= ChatPromptTemplate.from_messages([
        ("system", system_prompt),
        ("human", human_prompt)
    ])
    parser = StrOutputParser()
    return prompt|modelo|parser


In [3]:
from langchain_core.language_models.fake_chat_models import FakeListChatModel
chain = construir_chain(FakeListChatModel(responses=["positivo", "negativo", "positivo", "negativo", "positivo"]))

In [4]:
resultado1 = chain.invoke({"comentario": "O produto é bom, mas o atendimento é ruim"})
print(resultado1)
resultado2 = chain.invoke({"comentario": "O produto é bom, mas o atendimento é ruim"})
print(resultado2)

positivo
negativo


In [5]:
import sys
from pathlib import Path
RAIZ = Path.cwd()
if RAIZ.name == "notebook": RAIZ = RAIZ.parent
assert (RAIZ / "src").is_dir(), "Abra pela pasta da aula ou notebook"
sys.path.insert(0, str(RAIZ))
from src.dados import validar_rotulo
print(validar_rotulo(resultado1))

positivo


In [10]:
chain_teste = construir_chain(FakeListChatModel(responses=["positivo", "talvez", "negativo", "negativo"]))


In [9]:
saida_bruta = chain_teste.invoke({"comentario": "O produto é bom, mas o atendimento é ruim"})
#colocar o validador_rotulo em código try
print("Saída bruta:", saida_bruta)
try:
    previsto = validar_rotulo(saida_bruta)
    erro = None
except ValueError as e:
    previsto = None
    erro = str(e)
print("Previsão:", previsto)
print("Erro:", erro)

Saída bruta: positivo
Previsão: positivo
Erro: None


Plano: 1) rode com comentários sintéticos; 2) valide o rótulo; 3) conecte a API; 4) preserve id e texto; 5) registre saída bruta, previsão e erros; 6) confira os resultados manualmente. Não use histórico de outros comentários independentes.

In [61]:
resultados = []  # Preencha com as saídas da sua implementação.
dicts_resultados = {"id":"1", "texto":"O produto é bom, mas o atendimento é ruim",
                     "esperado":"positivo", "saida_bruta":saida_bruta, 
                     "previsto":previsto, "erro":erro}
resultados.append(dicts_resultados)
print(resultados)
# Campos sugeridos: id, texto, esperado, saida_bruta, previsto, erro.
# Não calcule métricas de qualidade sobre respostas do modelo simulado.

[{'id': '1', 'texto': 'O produto é bom, mas o atendimento é ruim', 'esperado': 'positivo', 'saida_bruta': 'negativo', 'previsto': 'negativo', 'erro': None}]


In [24]:
from langchain_ollama import ChatOllama
modelo_local = ChatOllama(
    base_url     = "http://localhost:11434",
    client_kwargs = {"trust_env": False},
    model        = "qwen2.5:3b",
    temperature  = 0
)
chain_local = construir_chain(modelo_local)

### Opção de diagnóstico: prompt simples

Execute esta célula depois de criar modelo_local. Ela prepara chain_simples com o mesmo modelo, mas sem as regras de maioria e desempate. Não executa uma classificação.

Para comparar, na célula da amostra troque somente chain_local.invoke por chain_simples.invoke. Preserve os comentários, os rótulos esperados e a validação. Esta versão diagnostica o reconhecimento do sentimento geral; não implementa a política completa.


In [25]:
def construir_chain_simples(modelo):
    """Monte uma chain curta para diagnosticar o comportamento do modelo."""
    prompt = ChatPromptTemplate.from_messages([
        ("system", "Classifique o sentimento geral do comentário. "
         "Responda somente positivo ou negativo, em minúsculas, "
         "sem explicações ou pontuação."),
        ("human", "Comentario: {comentario}"),
    ])
    parser = StrOutputParser()
    return prompt | modelo | parser


chain_simples = construir_chain_simples(modelo_local)


In [26]:
print("início")
amostra = [
    {"id":1,"texto":"O produto é excelente.","esperado": "positivo"},
    {"id":2,"texto":"O produto é ótimo, o atendimento foi péssimo e a entrega atrasou.","esperado": "negativo"},
    {"id":3,"texto":"O produto veio quebrado, mas o atendimento foi excelente.","esperado": "negativo"},
    {"id":4,"texto":"A entrega foi rápida, mas a embalagem veio danificada.","esperado": "negativo"},
    {"id":5,"texto":"entrega excelente.","esperado": "positivo"},
    {"id":6,"texto":"O produto é bom e o atendimento foi excelente.","esperado": "positivo"}
]
resultados = []
for item in amostra:
    id_avaliacao = item["id"]
    texto = item["texto"]
    esperado = item["esperado"]
    print(f"Processando avaliação {id_avaliacao}: {texto} Esperado: {esperado}")
    saida_bruta = chain_local.invoke({"comentario": texto})
    print("Saída bruta:", saida_bruta)
    try:
        previsto = validar_rotulo(saida_bruta)
        erro = None
    except ValueError as e:
        previsto = None
        erro = str(e)
    print("Previsão:", previsto)
    print("Erro:", erro)
    dicts_resultados = {"id": id_avaliacao, "texto": texto, "esperado": esperado, "saida_bruta": saida_bruta, "previsto": previsto, "erro": erro}
    resultados.append(dicts_resultados)
    print("Resultado adicionado:", dicts_resultados)
print("quantidade de resultados:", len(resultados))
print("Resultados finais:", resultados)


início
Processando avaliação 1: O produto é excelente. Esperado: positivo
Saída bruta: positivo
Previsão: positivo
Erro: None
Resultado adicionado: {'id': 1, 'texto': 'O produto é excelente.', 'esperado': 'positivo', 'saida_bruta': 'positivo', 'previsto': 'positivo', 'erro': None}
Processando avaliação 2: O produto é ótimo, o atendimento foi péssimo e a entrega atrasou. Esperado: negativo
Saída bruta: negativo
Previsão: negativo
Erro: None
Resultado adicionado: {'id': 2, 'texto': 'O produto é ótimo, o atendimento foi péssimo e a entrega atrasou.', 'esperado': 'negativo', 'saida_bruta': 'negativo', 'previsto': 'negativo', 'erro': None}
Processando avaliação 3: O produto veio quebrado, mas o atendimento foi excelente. Esperado: negativo
Saída bruta: negativo
Previsão: negativo
Erro: None
Resultado adicionado: {'id': 3, 'texto': 'O produto veio quebrado, mas o atendimento foi excelente.', 'esperado': 'negativo', 'saida_bruta': 'negativo', 'previsto': 'negativo', 'erro': None}
Processando 

### Registro do experimento: prompt detalhado e prompt simples

Comparação com o modelo local `llama3.2:1b`, temperatura zero e a mesma amostra de seis comentários. Foram mantidos o parser textual e a validação por `validar_rotulo`.

| Versão | Resultado observado |
|---|---|
| Prompt detalhado, com maioria e desempate | Cinco respostas `negativo`, incluindo dois elogios classificados incorretamente. No elogio à entrega, o modelo repetiu o comentário. Total: três classificações válidas e corretas, duas válidas incorretas e uma saída inválida. |
| Prompt simples, sobre sentimento geral | Quatro classificações válidas e corretas. Nos dois últimos comentários, retornou `sentimento geral: positivo`, rejeitado pelo formato. Total: quatro classificações válidas e corretas e duas saídas inválidas. |

A instrução curta melhorou o reconhecimento dos elogios nesta amostra, mas ainda não garantiu o formato exigido. A validação rejeitou as respostas com prefixo, preservando a saída bruta e o erro e atribuindo `None` à previsão.

O prompt simples não contém a política completa. A coincidência nos comentários mistos não comprova que ele segue as regras de maioria e desempate. Os exemplos já foram utilizados para orientar ajustes: este é um diagnóstico exploratório, não uma demonstração de qualidade geral.

Próximo experimento proposto: comparar os dois prompts com outro modelo, mantendo a amostra e separando falhas de formato de divergências na classificação.


Critérios de estudo: consigo explicar cada componente? A entrada vazia é tratada? Uma resposta fora do contrato é identificada? Há exemplos de negação e ambiguidade? O README permite outra pessoa executar?

Entrega no Moodle: somente link do repositório, até 31/10/2026 às 23h59, conforme a atividade. Consulte material/FONTES.md. LangGraph não aparece como exigência explícita no esqueleto consultado.